In [2]:
import scipy.io as sio
import pandas as pd
import numpy as np
import sys, os
import torch
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import Extract_Features

In [3]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

In [6]:

shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "rms_energy",
    frame_size = 36,
    hop_length = 36
)


print(data.get_samples().shape)
print(data.get_labels().shape)



(3309, 1001)
(3309,)


In [7]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel="rbf", C=100)
svc.fit(data.get_samples()[:3000], data.get_labels()[:3000])
print(svc.score(data.get_samples()[3000:], data.get_labels()[3000:]))

0.8284789644012945


In [8]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="./model_paths/rms_1.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-2, optim="adam", epochs=20)

loops.test(model_path="./model_paths/rms_1.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.581109, Train accuracy: 0.6630
[INFO] EPOCH: 2/20
Train loss: 0.519232, Train accuracy: 0.7173
[INFO] EPOCH: 3/20
Train loss: 0.497142, Train accuracy: 0.7433
[INFO] EPOCH: 4/20
Train loss: 0.487609, Train accuracy: 0.7493
[INFO] EPOCH: 5/20
Train loss: 0.479529, Train accuracy: 0.7547
[INFO] EPOCH: 6/20
Train loss: 0.472150, Train accuracy: 0.7690
[INFO] EPOCH: 7/20
Train loss: 0.466706, Train accuracy: 0.7733
[INFO] EPOCH: 8/20
Train loss: 0.461342, Train accuracy: 0.7777
[INFO] EPOCH: 9/20
Train loss: 0.452172, Train accuracy: 0.7777
[INFO] EPOCH: 10/20
Train loss: 0.450997, Train accuracy: 0.7807
[INFO] EPOCH: 11/20
Train loss: 0.448383, Train accuracy: 0.7810
[INFO] EPOCH: 12/20
Train loss: 0.441406, Train accuracy: 0.7910
[INFO] EPOCH: 13/20
Train loss: 0.437260, Train accuracy: 0.7880
[INFO] EPOCH: 14/20
Train loss: 0.433303, Train accuracy: 0.7940
[INFO] EPOCH: 15/20
Train loss: 0.429371, Train accuracy: 0.7923
[INFO] EPOCH: 16/20
Train loss: 0.

In [9]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_2_layer(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="./model_paths/rms_2.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-3, optim="adam", epochs=20)

loops.test(model_path="./model_paths/rms_2.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.568551, Train accuracy: 0.6803
[INFO] EPOCH: 2/20
Train loss: 0.498546, Train accuracy: 0.7437
[INFO] EPOCH: 3/20
Train loss: 0.469493, Train accuracy: 0.7600
[INFO] EPOCH: 4/20
Train loss: 0.460320, Train accuracy: 0.7657
[INFO] EPOCH: 5/20
Train loss: 0.437495, Train accuracy: 0.7867
[INFO] EPOCH: 6/20
Train loss: 0.435322, Train accuracy: 0.7910
[INFO] EPOCH: 7/20
Train loss: 0.418156, Train accuracy: 0.8037
[INFO] EPOCH: 8/20
Train loss: 0.412173, Train accuracy: 0.7963
[INFO] EPOCH: 9/20
Train loss: 0.404180, Train accuracy: 0.8063
[INFO] EPOCH: 10/20
Train loss: 0.393182, Train accuracy: 0.8077
[INFO] EPOCH: 11/20
Train loss: 0.410449, Train accuracy: 0.7977
[INFO] EPOCH: 12/20
Train loss: 0.378667, Train accuracy: 0.8237
[INFO] EPOCH: 13/20
Train loss: 0.375581, Train accuracy: 0.8257
[INFO] EPOCH: 14/20
Train loss: 0.367698, Train accuracy: 0.8293
[INFO] EPOCH: 15/20
Train loss: 0.361411, Train accuracy: 0.8297
[INFO] EPOCH: 16/20
Train loss: 0.

In [11]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3309))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP_3_layer(nb_hidden=512, input_dim=data.get_samples().shape[1], output_dim=2, dropout_rate=0.5)

loops.train(model=model, model_path="./model_paths/rms_3.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=1e-2, optim="adam", epochs=20)

loops.test(model_path="./model_paths/rms_3.pth", test_loader=test_loader)

[INFO] EPOCH: 1/20
Train loss: 0.566289, Train accuracy: 0.6770
[INFO] EPOCH: 2/20
Train loss: 0.505571, Train accuracy: 0.7347
[INFO] EPOCH: 3/20
Train loss: 0.477615, Train accuracy: 0.7560
[INFO] EPOCH: 4/20
Train loss: 0.457336, Train accuracy: 0.7783
[INFO] EPOCH: 5/20
Train loss: 0.446582, Train accuracy: 0.7860
[INFO] EPOCH: 6/20
Train loss: 0.441847, Train accuracy: 0.7843
[INFO] EPOCH: 7/20
Train loss: 0.425714, Train accuracy: 0.8003
[INFO] EPOCH: 8/20
Train loss: 0.414558, Train accuracy: 0.8050
[INFO] EPOCH: 9/20
Train loss: 0.416273, Train accuracy: 0.7970
[INFO] EPOCH: 10/20
Train loss: 0.399629, Train accuracy: 0.8077
[INFO] EPOCH: 11/20
Train loss: 0.395533, Train accuracy: 0.8077
[INFO] EPOCH: 12/20
Train loss: 0.391826, Train accuracy: 0.8130
[INFO] EPOCH: 13/20
Train loss: 0.385582, Train accuracy: 0.8193
[INFO] EPOCH: 14/20
Train loss: 0.358228, Train accuracy: 0.8367
[INFO] EPOCH: 15/20
Train loss: 0.349043, Train accuracy: 0.8383
[INFO] EPOCH: 16/20
Train loss: 0.